# Qwen2-VL Fine-tuning on DocVQA

Notebook for Google Colab A100. It fine-tunes both `Qwen/Qwen2-VL-2B-Instruct` and `Qwen/Qwen2-VL-7B-Instruct` sequentially with QLoRA, logs metrics and tables to Comet, and uploads adapter checkpoints to Hugging Face Hub.

The chat formatting follows the official Qwen2-VL processor/chat-template workflow from the Hugging Face Transformers docs, using `processor.apply_chat_template(...)` and multimodal processing through `Qwen2VLProcessor`.

Reference: https://huggingface.co/docs/transformers/model_doc/qwen2_vl

## 0. Setup

In [ ]:
%pip install -q huggingface_hub comet_ml python-dotenv peft bitsandbytes trl qwen-vl-utils

In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


project_root = find_project_root(Path.cwd())
repo_url = get_colab_secret('COURSE_WORK2026_REPO_URL')

if project_root is None:
    clone_dir = Path('/content/course_work2026')
    if not (clone_dir / '.git').exists():
        if not repo_url:
            raise RuntimeError('Could not detect the repository locally and COURSE_WORK2026_REPO_URL is not set in Colab secrets.')
        subprocess.run(['git', 'clone', repo_url, str(clone_dir)], check=True)
    project_root = clone_dir

sys.path.insert(0, str(project_root / 'src'))

from colab_setup import setup_colab

setup_summary, bootstrap_experiment = setup_colab(repo_url=repo_url)
bootstrap_experiment.set_name('qwen2vl-bootstrap')
bootstrap_experiment.end()

PROJECT_ROOT = Path(setup_summary['project_root'])
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
TRAIN_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'train_qwen2vl.jsonl'
VALIDATION_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'validation_qwen2vl.jsonl'
OUTPUT_ROOT = ARTIFACTS_ROOT / 'finetuning_generative' / 'qwen2vl_colab'
HF_CHECKPOINT_REPO = os.getenv('HF_MODEL_REPO') or 'karimkramin/docvqa-privacy-checkpoints'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
print({'project_root': str(PROJECT_ROOT), 'train_jsonl_exists': TRAIN_JSONL.exists(), 'validation_jsonl_exists': VALIDATION_JSONL.exists(), 'hf_checkpoint_repo': HF_CHECKPOINT_REPO})

## 1. Config

In [ ]:
import math
import re
import time
from dataclasses import dataclass
from typing import Any

import bitsandbytes as bnb
import comet_ml
import pandas as pd
import torch
from huggingface_hub import HfApi
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

MODEL_CONFIGS = [
    {'name': 'Qwen/Qwen2-VL-2B-Instruct', 'tag': 'qwen2b', 'epochs': 30, 'batch_size': 2, 'checkpoint_epochs': [1, 3, 10, 30]},
    {'name': 'Qwen/Qwen2-VL-7B-Instruct', 'tag': 'qwen7b', 'epochs': 30, 'batch_size': 1, 'checkpoint_epochs': [1, 3, 10, 30]},
]
LORA_R, LORA_ALPHA, LR, GRADIENT_ACCUMULATION = 16, 32, 2e-4, 8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
api = HfApi()
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type='model', private=True, exist_ok=True)

print({'device': str(device), 'model_configs': MODEL_CONFIGS, 'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'lr': LR, 'gradient_accumulation': GRADIENT_ACCUMULATION})

## 2-5. Train Both Models Sequentially

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(raw_path: str, split: str) -> Path:
    original = Path(raw_path)
    if original.exists():
        return original
    benchmark_dir = 'benchmark_train' if split == 'train' else 'benchmark'
    candidate = ARTIFACTS_ROOT / 'docqa_recovery' / benchmark_dir / 'images' / 'original' / original.name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Cannot resolve image path for {raw_path}')


def normalize_qwen_row(row: dict[str, Any]) -> dict[str, Any]:
    row = dict(row)
    row['image_path'] = str(resolve_image_path(str(row['image_path']), str(row['split'])))
    chat_messages = row['chat_messages']
    normalized_messages = []
    for message in chat_messages:
        normalized_content = []
        for content_item in message['content']:
            item = dict(content_item)
            if item.get('type') == 'image':
                item['image'] = str(resolve_image_path(str(item['image']), str(row['split'])))
            normalized_content.append(item)
        normalized_messages.append({'role': message['role'], 'content': normalized_content})
    row['chat_messages'] = normalized_messages
    return row


TRAIN_ROWS = [normalize_qwen_row(row) for row in load_jsonl(TRAIN_JSONL)]
VALIDATION_ROWS = [normalize_qwen_row(row) for row in load_jsonl(VALIDATION_JSONL)]


class QwenChatDataset(Dataset):
    def __init__(self, rows: list[dict[str, Any]]):
        self.rows = rows

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.rows[idx]


def make_chat_collator(processor):
    def collate_fn(batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        prompt_messages = []
        full_messages = []
        images = []

        for row in batch:
            user_message = row['chat_messages'][0]
            assistant_message = row['chat_messages'][1]
            prompt_messages.append([user_message])
            full_messages.append([user_message, assistant_message])
            images.append(Image.open(row['image_path']).convert('RGB'))

        prompt_texts = [processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) for msgs in prompt_messages]
        full_texts = [processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False) for msgs in full_messages]

        prompt_image_inputs = []
        full_image_inputs = []
        for msgs in prompt_messages:
            image_inputs, _video_inputs = process_vision_info(msgs)
            prompt_image_inputs.append(image_inputs[0] if isinstance(image_inputs, list) else image_inputs)
        for msgs in full_messages:
            image_inputs, _video_inputs = process_vision_info(msgs)
            full_image_inputs.append(image_inputs[0] if isinstance(image_inputs, list) else image_inputs)

        prompt_batch = processor(text=prompt_texts, images=prompt_image_inputs, padding=True, return_tensors='pt')
        full_batch = processor(text=full_texts, images=full_image_inputs, padding=True, return_tensors='pt')

        labels = full_batch['input_ids'].clone()
        labels[full_batch['attention_mask'] == 0] = -100
        prompt_lengths = prompt_batch['attention_mask'].sum(dim=1).tolist()
        for idx, prompt_len in enumerate(prompt_lengths):
            labels[idx, : int(prompt_len)] = -100

        batch_dict = {k: v for k, v in full_batch.items()}
        batch_dict['labels'] = labels
        return batch_dict

    return collate_fn


def find_lora_target_modules(model) -> list[str]:
    linear_class = bnb.nn.Linear4bit
    discovered = []
    for module_name, module in model.named_modules():
        if isinstance(module, linear_class):
            discovered.append(module_name.split('.')[-1])
    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    unique_discovered = sorted(set(discovered))
    selected = [name for name in preferred if name in unique_discovered]
    if not selected:
        selected = unique_discovered
    print({'discovered_linear_modules': unique_discovered, 'selected_lora_target_modules': selected})
    return selected


def trainable_params_pct(model) -> float:
    total = 0
    trainable = 0
    for param in model.parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    return 100.0 * trainable / max(total, 1)


class CometCheckpointCallback(TrainerCallback):
    def __init__(self, experiment, processor, tag: str, checkpoint_epochs: list[int], checkpoint_root: Path, hf_repo_id: str, hf_api: HfApi):
        self.experiment = experiment
        self.processor = processor
        self.tag = tag
        self.checkpoint_epochs = set(checkpoint_epochs)
        self.checkpoint_root = checkpoint_root
        self.hf_repo_id = hf_repo_id
        self.hf_api = hf_api
        self.current_epoch_losses: list[float] = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if 'loss' in logs:
            self.current_epoch_losses.append(float(logs['loss']))

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(round(state.epoch or 0))
        if self.current_epoch_losses:
            mean_loss = sum(self.current_epoch_losses) / len(self.current_epoch_losses)
            self.experiment.log_metric('train_loss', mean_loss, epoch=epoch)
            self.current_epoch_losses = []

        if epoch in self.checkpoint_epochs:
            model = kwargs['model']
            checkpoint_path = self.checkpoint_root / f'epoch_{epoch}'
            checkpoint_path.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(checkpoint_path)
            self.processor.save_pretrained(checkpoint_path)
            self.hf_api.upload_folder(
                folder_path=str(checkpoint_path),
                repo_id=self.hf_repo_id,
                repo_type='model',
                path_in_repo=f'{self.tag}/epoch_{epoch}',
            )
            self.experiment.log_metric('checkpoint_saved', epoch, epoch=epoch)
            print(f'Uploaded adapter checkpoint: {checkpoint_path}')


def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())


def sanitize_prediction_text(text: str) -> str:
    text = str(text or '')
    text = re.sub(r'<[^>]+>', ' ', text)
    text = ' '.join(text.replace('\n', ' ').split()).strip()
    return text


def exact_match(gold: str, pred: str) -> float:
    return float(normalize_text(gold) == normalize_text(pred))


def run_sanity_check(base_model_name: str, adapter_dir: Path, processor, rows: list[dict[str, Any]], split_name: str, limit: int = 100) -> tuple[pd.DataFrame, float]:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        base_model_name,
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    results = []
    for row in rows[:limit]:
        user_messages = [row['chat_messages'][0]]
        prompt_text = processor.apply_chat_template(user_messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _video_inputs = process_vision_info(user_messages)
        image_input = image_inputs[0] if isinstance(image_inputs, list) else image_inputs
        inputs = processor(text=[prompt_text], images=[image_input], padding=True, return_tensors='pt')
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

        input_len = inputs['input_ids'].shape[1]
        output_ids = generated_ids[:, input_len:]
        decoded = processor.batch_decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]
        prediction = sanitize_prediction_text(decoded)
        question_text = ''
        for content_item in row['chat_messages'][0]['content']:
            if content_item.get('type') == 'text':
                question_text = content_item.get('text', '')
                break
        results.append({
            'split': split_name,
            'example_id': row['example_id'],
            'question': question_text,
            'gold': row['answer'],
            'prediction': prediction,
            'exact_match': exact_match(row['answer'], prediction),
        })

    df = pd.DataFrame(results)
    em = float(df['exact_match'].mean()) if not df.empty else 0.0
    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()
    return df, em


for config in MODEL_CONFIGS:
    model_name = config['name']
    tag = config['tag']
    checkpoint_epochs = config['checkpoint_epochs']
    batch_size = config['batch_size']
    num_epochs = config['epochs']

    experiment = comet_ml.Experiment(project_name='docvqa-privacy')
    experiment.set_name(f'{tag}-finetune')

    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    target_modules = find_lora_target_modules(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)
    trainable_pct = trainable_params_pct(model)
    model.print_trainable_parameters()
    experiment.log_parameter('trainable_params_pct', trainable_pct)
    experiment.log_parameters({
        'model_name': model_name,
        'tag': tag,
        'epochs': num_epochs,
        'batch_size': batch_size,
        'gradient_accumulation': GRADIENT_ACCUMULATION,
        'learning_rate': LR,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'target_modules': ','.join(target_modules),
        'train_size': len(TRAIN_ROWS),
        'validation_size': len(VALIDATION_ROWS),
    })

    train_dataset = QwenChatDataset(TRAIN_ROWS)
    data_collator = make_chat_collator(processor)
    model_output_root = OUTPUT_ROOT / tag
    checkpoint_root = model_output_root / 'checkpoints'
    checkpoint_root.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(model_output_root),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LR,
        logging_steps=10,
        save_strategy='no',
        evaluation_strategy='no',
        remove_unused_columns=False,
        bf16=False,
        fp16=torch.cuda.is_available(),
        report_to=[],
        optim='paged_adamw_8bit',
        dataloader_num_workers=2,
    )

    callback = CometCheckpointCallback(
        experiment=experiment,
        processor=processor,
        tag=tag,
        checkpoint_epochs=checkpoint_epochs,
        checkpoint_root=checkpoint_root,
        hf_repo_id=HF_CHECKPOINT_REPO,
        hf_api=api,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        callbacks=[callback],
    )

    start_time = time.time()
    trainer.train()
    experiment.log_metric('train_runtime_seconds', time.time() - start_time)

    final_epoch_dir = checkpoint_root / f"epoch_{num_epochs}"
    if not final_epoch_dir.exists():
        final_epoch_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(final_epoch_dir)
        processor.save_pretrained(final_epoch_dir)

    seen_df, em_seen = run_sanity_check(model_name, final_epoch_dir, processor, TRAIN_ROWS, 'seen_train', limit=100)
    unseen_df, em_unseen = run_sanity_check(model_name, final_epoch_dir, processor, VALIDATION_ROWS, 'unseen_validation', limit=100)
    sanity_df = pd.concat([seen_df, unseen_df], ignore_index=True)
    sanity_csv_path = model_output_root / 'sanity_check.csv'
    sanity_df.to_csv(sanity_csv_path, index=False)

    experiment.log_metrics({'em_seen': em_seen, 'em_unseen': em_unseen})
    experiment.log_table('sanity_check.csv', sanity_df)
    display(sanity_df[['split', 'question', 'gold', 'prediction', 'exact_match']].head(10))
    print({'tag': tag, 'em_seen': em_seen, 'em_unseen': em_unseen, 'sanity_csv_path': str(sanity_csv_path)})

    experiment.end()
    del trainer
    del model
    del processor
    del train_dataset
    del data_collator
    del callback
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Completed run for {tag}')